# Factorizations and other fun
Based on work by Andreas Noack

## Outline
 - Factorizations
 - Special matrix structures
 - Generic linear algebra

Before we get started, let's set up a linear system and use `LinearAlgebra` to bring in the factorizations and special matrix structures.

In [10]:
using LinearAlgebra
A = rand(3, 3)
x = fill(1, (3,))
b = A * x

3-element Vector{Float64}:
 1.6329809677927671
 1.3190119479773177
 1.3962289898672258

In [11]:
A

3×3 Matrix{Float64}:
 0.679515  0.437643  0.515823
 0.223554  0.575268  0.52019
 0.422847  0.316385  0.656998

In [12]:
x

3-element Vector{Int64}:
 1
 1
 1

In [13]:
b

3-element Vector{Float64}:
 1.6329809677927671
 1.3190119479773177
 1.3962289898672258

## Factorizations

#### LU factorizations
In Julia we can perform an LU factorization
```julia
PA = LU
``` 
where `P` is a permutation matrix, `L` is lower triangular unit diagonal and `U` is upper triangular, using `lufact`.

Julia allows computing the LU factorization and defines a composite factorization type for storing it.

In [14]:
Alu = lu(A)

LU{Float64, Matrix{Float64}, Vector{Int64}}
L factor:
3×3 Matrix{Float64}:
 1.0       0.0       0.0
 0.328991  1.0       0.0
 0.622278  0.102134  1.0
U factor:
3×3 Matrix{Float64}:
 0.679515  0.437643  0.515823
 0.0       0.431287  0.350488
 0.0       0.0       0.300216

In [15]:
typeof(Alu)

LU{Float64, Matrix{Float64}, Vector{Int64}}

The different parts of the factorization can be extracted by accessing their special properties

In [16]:
Alu.P

3×3 Matrix{Float64}:
 1.0  0.0  0.0
 0.0  1.0  0.0
 0.0  0.0  1.0

In [17]:
Alu.L

3×3 Matrix{Float64}:
 1.0       0.0       0.0
 0.328991  1.0       0.0
 0.622278  0.102134  1.0

In [18]:
Alu.U

3×3 Matrix{Float64}:
 0.679515  0.437643  0.515823
 0.0       0.431287  0.350488
 0.0       0.0       0.300216

Julia can dispatch methods on factorization objects.

For example, we can solve the linear system using either the original matrix or the factorization object.

In [19]:
A\b

3-element Vector{Float64}:
 1.0
 1.0
 1.0

In [20]:
Alu\b

3-element Vector{Float64}:
 1.0
 1.0
 1.0

Similarly, we can calculate the determinant of `A` using either `A` or the factorization object

In [21]:
det(A) ≈ det(Alu)

true

#### QR factorizations

In Julia we can perform a QR factorization
```
A=QR
``` 

where `Q` is unitary/orthogonal and `R` is upper triangular, using `qrfact`. 

In [22]:
Aqr = qr(A)

LinearAlgebra.QRCompactWY{Float64, Matrix{Float64}, Matrix{Float64}}
Q factor: 3×3 LinearAlgebra.QRCompactWYQ{Float64, Matrix{Float64}, Matrix{Float64}}
R factor:
3×3 Matrix{Float64}:
 -0.830973  -0.673633  -0.89607
  0.0       -0.410831  -0.314574
  0.0        0.0        0.25772

Similarly to the LU factorization, the matrices `Q` and `R` can be extracted from the QR factorization object via

In [23]:
Aqr.Q

3×3 LinearAlgebra.QRCompactWYQ{Float64, Matrix{Float64}, Matrix{Float64}}

In [24]:
Aqr.R

3×3 Matrix{Float64}:
 -0.830973  -0.673633  -0.89607
  0.0       -0.410831  -0.314574
  0.0        0.0        0.25772

#### Eigendecompositions

The results from eigendecompositions, singular value decompositions, Hessenberg factorizations, and Schur decompositions are all stored in `Factorization` types.

The eigendecomposition can be computed

In [25]:
Asym = A + A'
AsymEig = eigen(Asym)

Eigen{Float64, Float64, Matrix{Float64}, Vector{Float64}}
values:
3-element Vector{Float64}:
 0.32388402695980334
 0.5869132004431913
 2.912763120751171
vectors:
3×3 Matrix{Float64}:
 -0.417371   0.688571   0.593019
 -0.45955   -0.722911   0.515958
  0.783974  -0.0571761  0.618155

The values and the vectors can be extracted from the Eigen type by special indexing

In [26]:
AsymEig.values

3-element Vector{Float64}:
 0.32388402695980334
 0.5869132004431913
 2.912763120751171

In [27]:
AsymEig.vectors

3×3 Matrix{Float64}:
 -0.417371   0.688571   0.593019
 -0.45955   -0.722911   0.515958
  0.783974  -0.0571761  0.618155

Once again, when the factorization is stored in a type, we can dispatch on it and write specialized methods that exploit the properties of the factorization, e.g. that $A^{-1}=(V\Lambda V^{-1})^{-1}=V\Lambda^{-1}V^{-1}$.

In [28]:
inv(AsymEig)*Asym

3×3 Matrix{Float64}:
  1.0          -2.22045e-16   0.0
 -7.77156e-16   1.0          -2.22045e-16
  6.66134e-16   1.11022e-15   1.0

## Special matrix structures
Matrix structure is very important in linear algebra. To see *how* important it is, let's work with a larger linear system

In [29]:
n = 1000
A = randn(n,n);

In [31]:
A

1000×1000 Matrix{Float64}:
  0.160868    0.126601   -0.90738    …   1.23266     0.615691    1.40741
  1.60796    -0.335353    0.574073       0.711768   -0.312688   -0.0133509
  0.0523852  -0.0984632   0.872638      -1.67187     0.166214    0.320004
  1.41155     2.02371    -0.76274       -1.09706     0.0524632   0.195292
 -0.363511   -1.81216     0.0640831      0.614064    0.517517    0.808805
 -0.0348255   0.5052      0.324985   …  -0.347145   -0.459192   -0.562513
  0.540973    1.47375    -0.777001       0.255278   -0.891258   -0.432008
 -0.137441   -0.0806766   0.409071      -0.72958    -0.449083    1.55768
 -0.497733   -0.556712   -1.70433        0.748792    0.0829784  -0.134727
 -1.33374     0.463438   -0.525047      -0.134265    2.37719    -0.72397
  ⋮                                  ⋱                          
 -0.817004   -0.195605    0.834161      -0.878153    2.13597     0.923499
  0.657653   -0.0602792  -2.06116       -0.427562    1.35776     0.715341
 -0.491212    0.59063 

Julia can often infer special matrix structure

In [32]:
Asym = A + A'
issymmetric(Asym)

true

but sometimes floating point error might get in the way.

In [33]:
Asym_noisy = copy(Asym)
Asym_noisy[1,2] += 5eps()

1.7345585555567355

In [37]:
3 * eps()

6.661338147750939e-16

In [38]:
issymmetric(Asym_noisy)

false

Luckily we can declare structure explicitly with, for example, `Diagonal`, `Triangular`, `Symmetric`, `Hermitian`, `Tridiagonal` and `SymTridiagonal`.

In [42]:
Asym_explicit = Symmetric(Asym_noisy);

Let's compare how long it takes Julia to compute the eigenvalues of `Asym`, `Asym_noisy`, and `Asym_explicit`

In [43]:
@time eigvals(Asym);

  1.195557 seconds (2.70 M allocations: 142.238 MiB, 13.81% gc time, 94.16% compilation time)


In [23]:
@time eigvals(Asym_noisy);

  1.050047 seconds (13 allocations: 7.920 MiB)


In [24]:
@time eigvals(Asym_explicit);

  0.149938 seconds (5.88 k allocations: 8.358 MiB, 2.11% compilation time)


In this example, using `Symmetric()` on `Asym_noisy` made our calculations about `5x` more efficient :)

#### A big problem
Using the `Tridiagonal` and `SymTridiagonal` types to store tridiagonal matrices makes it possible to work with potentially very large tridiagonal problems. The following problem would not be possible to solve on a laptop if the matrix had to be stored as a (dense) `Matrix` type.

In [44]:
n = 1_000_000;
A = SymTridiagonal(randn(n), randn(n-1));
@time eigmax(A)

  0.840105 seconds (1.04 M allocations: 234.807 MiB, 20.87% gc time, 32.83% compilation time)


6.346350596121866

## Generic linear algebra
The usual way of adding support for numerical linear algebra is by wrapping BLAS and LAPACK subroutines. For matrices with elements of `Float32`, `Float64`, `Complex{Float32}` or `Complex{Float64}` this is also what Julia does.

However, Julia also supports generic linear algebra, allowing you to, for example, work with matrices and vectors of rational numbers.

#### Rational numbers
Julia has rational numbers built in. To construct a rational number, use double forward slashes:

In [45]:
1//2

1//2

#### Example: Rational linear system of equations
The following example shows how linear system of equations with rational elements can be solved without promoting to floating point element types. Overflow can easily become a problem when working with rational numbers so we use `BigInt`s.

In [46]:
Arational = Matrix{Rational{BigInt}}(rand(1:10, 3, 3))/10

3×3 Matrix{Rational{BigInt}}:
 2//5    1     3//10
 3//10  1//10  3//5
 4//5    1     3//5

In [47]:
x = fill(1, 3)
b = Arational*x

3-element Vector{Rational{BigInt}}:
 17//10
   1
 12//5

In [48]:
Arational\b

3-element Vector{Rational{BigInt}}:
 1
 1
 1

In [49]:
lu(Arational)

LU{Rational{BigInt}, Matrix{Rational{BigInt}}, Vector{Int64}}
L factor:
3×3 Matrix{Rational{BigInt}}:
  1       0     0
 1//2     1     0
 3//8  -11//20  1
U factor:
3×3 Matrix{Rational{BigInt}}:
 4//5   1    3//5
  0    1//2   0
  0     0    3//8

### Exercises

#### 11.1
What are the eigenvalues of matrix A?

```
A =
[
 140   97   74  168  131
  97  106   89  131   36
  74   89  152  144   71
 168  131  144   54  142
 131   36   71  142   36
]
```
and assign it a variable `A_eigv`

In [ ]:
using LinearAlgebra

In [ ]:
@assert A_eigv ==  [-128.49322764802145, -55.887784553056875, 42.7521672793189, 87.16111477514521, 542.4677301466143]

#### 11.2 
Create a `Diagonal` matrix from the eigenvalues of `A`.

In [ ]:
@assert A_diag ==  [-128.493    0.0      0.0      0.0       0.0;
    0.0    -55.8878   0.0      0.0       0.0;
    0.0      0.0     42.7522   0.0       0.0;
    0.0      0.0      0.0     87.1611    0.0;
    0.0 0.0      0.0      0.0     542.468]

#### 11.3 
Create a `LowerTriangular` matrix from `A` and store it in `A_lowertri`

In [ ]:
@assert A_lowertri ==  [140    0    0    0   0;
  97  106    0    0   0;
  74   89  152    0   0;
 168  131  144   54   0;
 131   36   71  142  36]